# Generic Binary Guardrail Tuning

This notebook demonstrates the generic `BinaryGuardrailTuner` API. The example uses `PromptInjection`, but the same pattern works for any guardrail that classifies inputs into valid (`True`) or invalid (`False`).

You only provide CSV data with these columns:
- `text`: the text to scan
- `label`: `True` means valid, `False` means invalid

The scanner class owns its optimization space, and the tuner picks the default optimizer for you. You can optionally switch the optimizer by name.

If a scanner needs fixed non-optimized arguments, provide them through `SCANNER_FIXED_KWARGS`.

In [1]:
import csv
import json
from pathlib import Path

from dotenv import load_dotenv

from testsavant.guard import BinaryGuardrailTuner
from testsavant.guard.input_scanners import PromptInjection

load_dotenv()

True

In [ ]:
TRAIN_CSV_PATH = Path('./sampled.csv')
TEST_CSV_PATH = None  
TEXT_COLUMN = 'text'
LABEL_COLUMN = 'label'
SCANNER_CLS = PromptInjection
SCANNER_FIXED_KWARGS = {}  # Example: {'topics': ['finance'], 'mode': 'blacklist'}
OPTIMIZER_NAME = 'optuna'
MAX_TRAIN_ROWS = None
MAX_TEST_ROWS = None
RANDOM_STATE = 42
EPOCHS = 2  # Uses scanner defaults when None
BATCH_SIZE = 16  # Uses scanner defaults when None
PATIENCE = 2 # 

In [4]:
scanner_defaults = SCANNER_CLS.get_optimization_defaults()
resolved_epochs = EPOCHS if EPOCHS is not None else int(scanner_defaults.get('epochs', 8))
resolved_batch_size = BATCH_SIZE if BATCH_SIZE is not None else int(scanner_defaults.get('batch_size', 32))
resolved_top_k = int(scanner_defaults.get('top_k', 10))

print({
    'scanner': SCANNER_CLS.__name__,
    'optimizer_name': OPTIMIZER_NAME,
    'epochs': resolved_epochs,
    'batch_size': resolved_batch_size,
    'top_k': resolved_top_k,
})

{'scanner': 'PromptInjection', 'optimizer_name': 'optuna', 'epochs': 2, 'batch_size': 16, 'top_k': 10}


In [5]:
def parse_valid_label(value):
    if isinstance(value, bool):
        return value

    normalized = str(value).strip().lower()
    if normalized in {'true', '1', 'yes', 'y', 'valid'}:
        return True
    if normalized in {'false', '0', 'no', 'n', 'invalid'}:
        return False
    raise ValueError(f'Unsupported label value: {value!r}')


def load_csv_dataset(path: Path, text_column: str, label_column: str, max_rows=None):
    rows = []
    with path.open('r', encoding='utf-8', newline='') as csv_file:
        reader = csv.DictReader(csv_file)
        for raw_row in reader:
            text = raw_row[text_column]
            label = parse_valid_label(raw_row[label_column])
            rows.append((text, label))
            if max_rows is not None and len(rows) >= max_rows:
                break
    return rows


train_rows = load_csv_dataset(TRAIN_CSV_PATH, TEXT_COLUMN, LABEL_COLUMN, max_rows=MAX_TRAIN_ROWS)
# split to train to test if no test csv is provided
test_rows = None
if TEST_CSV_PATH is not None:
    test_rows = load_csv_dataset(TEST_CSV_PATH, TEXT_COLUMN, LABEL_COLUMN, max_rows=MAX_TEST_ROWS)
else:
    # split to train to test if no test csv is provided, 
    split_index = int(0.7 * len(train_rows))
    test_rows = train_rows[split_index:]
    train_rows = train_rows[:split_index]

print({
    'train_rows': len(train_rows),
    'test_rows': len(test_rows) if test_rows is not None else 0,
})

{'train_rows': 70, 'test_rows': 30}


In [6]:
tuner = BinaryGuardrailTuner.from_input_scanner_class(
    scanner_cls=SCANNER_CLS,
    fixed_scanner_kwargs=SCANNER_FIXED_KWARGS,
    optimizer_name=OPTIMIZER_NAME,
    top_k=resolved_top_k,
    random_state=RANDOM_STATE,
)

tuner

In [7]:
def on_step(payload):
    if not payload['top_results']:
        return
    best_result = payload['top_results'][0]
    best_test_selection_score = payload.get('best_test_selection_score')
    best_test_score_text = (
        f" best_test_selection_score={best_test_selection_score:.4f}"
        if best_test_selection_score is not None
        else ""
    )
    print(
        f"step={payload['step']}/{payload['total_steps']} "
        f"best_mean_score={best_result['mean_score']:.4f} "
        f"best_objective_score={best_result['objective_score']:.4f}"
        f"{best_test_score_text} "
        f"config={best_result['config']}"
    )


fit_kwargs = {
    'train_x': [text for text, _ in train_rows],
    'train_y': [label for _, label in train_rows],
    'epochs': resolved_epochs,
    'batch_size': resolved_batch_size,
    'patience': PATIENCE,
    'on_step': on_step,
}

if test_rows is not None:
    fit_kwargs['test_x'] = [text for text, _ in test_rows]
    fit_kwargs['test_y'] = [label for _, label in test_rows]

fit_result = tuner.fit(**fit_kwargs)

fit_result.best_config

step=1/10 best_mean_score=0.6111 best_objective_score=0.6185 best_test_selection_score=0.7664 config={'threshold': 0.675, 'chunk_size': 250, 'overlap': 40}
step=2/10 best_mean_score=0.6111 best_objective_score=0.6185 best_test_selection_score=0.7664 config={'threshold': 0.675, 'chunk_size': 250, 'overlap': 40}
step=3/10 best_mean_score=0.6111 best_objective_score=0.6185 best_test_selection_score=0.7664 config={'threshold': 0.675, 'chunk_size': 250, 'overlap': 40}


{'threshold': 0.675, 'chunk_size': 250, 'overlap': 40}

In [9]:
train_result = tuner.evaluate(
    fit_result.best_config,
    [text for text, _ in train_rows],
    [label for _, label in train_rows],
)

report = {
    'best_config': fit_result.best_config,
    'train_result': train_result.to_dict(),
}

if test_rows is not None:
    test_result = tuner.evaluate(
        fit_result.best_config,
        [text for text, _ in test_rows],
        [label for _, label in test_rows],
    )
    combined_rows = train_rows + test_rows
    combined_result = tuner.evaluate(
        fit_result.best_config,
        [text for text, _ in combined_rows],
        [label for _, label in combined_rows],
    )
    report['test_result'] = test_result.to_dict()
    report['combined_result'] = combined_result.to_dict()
else:
    report['combined_result'] = train_result.to_dict()

print(json.dumps(report, indent=2))

{
  "best_config": {
    "threshold": 0.675,
    "chunk_size": 250,
    "overlap": 40
  },
  "train_result": {
    "total_count": 70,
    "valid_count": 15,
    "invalid_count": 55,
    "true_positive": 36,
    "false_positive": 12,
    "true_negative": 3,
    "false_negative": 19,
    "recall": 0.6545454545454545,
    "specificity": 0.2,
    "precision": 0.75,
    "f1_score": 0.6990291262135923,
    "false_positive_rate": 0.8,
    "false_negative_rate": 0.34545454545454546,
    "accuracy": 0.5571428571428572,
    "effectiveness_score": 0.3063829787234043,
    "selection_score": 0.3063829787234043,
    "predicted_valid_count": 22,
    "predicted_invalid_count": 48
  },
  "test_result": {
    "total_count": 30,
    "valid_count": 8,
    "invalid_count": 22,
    "true_positive": 15,
    "false_positive": 1,
    "true_negative": 7,
    "false_negative": 7,
    "recall": 0.6818181818181818,
    "specificity": 0.875,
    "precision": 0.9375,
    "f1_score": 0.7894736842105263,
    "false_po